# Blackboard DEMO

## Setup

In [ ]:
from src.board import Board

question = """
г. Тюмень социально-экономический анализ:
- демография,
- производительность труда, 
- структура и динамика ВГП,
- рынок труда, 
- рынок жилья и коммерческой недвижимости,
- бюджетная обеспеченность.
"""

board = Board(question=question)

In [3]:
from src.agents.factories import *

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [4]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(str.join(', ', expert.info.values()))

079fc8, Экономист‑аналитик регионального развития, Проводит комплексный социально‑экономический анализ регионов: оценивает структуру и динамику ВВП, производительность труда, рынок труда, а т
adc41b, Специалист по демографии и рынку недвижимости, Анализирует демографические тенденции, миграционные потоки, спрос и предложение на жильё и коммерческую недвижимость, их влияние на экономию
3e8d3d, Градостроительный планировщик‑экономист, Оценивает пространственное распределение экономической активности, взаимодействие инфраструктуры, бюджетную обеспеченность и их влияние на 


In [5]:
# from src.searcher import WikipediaSearcher
# wikipedia_searcher = WikipediaSearcher()

planner_agent = create_planner_agent(board)
# wikipedia_agent = create_wikipedia_agent(wikipedia_searcher, board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)
decider_agent = create_decider_agent(board)

role_agents = [planner_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent] # wikipedia_agent, 
controller_agent = create_controller_agent(role_agents, board)

## Experiment

In [6]:
from src.agents import Agent

def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

In [ ]:
from tqdm import tqdm
from langchain.messages import AIMessage, HumanMessage, SystemMessage

is_final = False

def get_invoke_message() -> SystemMessage:
    return SystemMessage(f"Записей на доске: {len(board.notes)}")

for i in range(1):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke([get_invoke_message()], force=True).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join(', ', [a.role.name for a in agents]))

    for a in tqdm(agents):
        try:
            response = a.invoke([get_invoke_message()], force=False)
        except:
            print(a.role.name + ' перегрелся')
            continue

        if a == decider_agent: # DECIDER AGENT
            note = response.note
            is_final = response.is_final
        elif a == cleaner_agent: # CLEANER AGENT
            notes_ids = response.notes_ids
            board.remove_notes(notes_ids)
            continue
        else:
            note = response

        board.add_note(note.content, a.id, a.role.name)

        if is_final:
            break

    if is_final:
        break

Итерация 1
Планировщик, Экономист‑аналитик регионального развития, Специалист по демографии и рынку недвижимости, Градостроительный планировщик‑экономист, Критик, Уборщик, Арбитр


 86%|████████▌ | 6/7 [05:24<00:49, 49.50s/it]

Уборщик перегрелся


In [7]:
board.print()

╭───────────────────── 📌#92d031 by a39afb (Региональный планировщик‑географ) ─────────────────────╮
│ ### Запись № 1 (ID: a39afb‑01)                                                                   │
│ **Тема:** План диагностики экономического и пространственного состояния г. Тюмень                │
│ **Автор:** Региональный планировщик‑географ (ID a39afb)                                          │
│                                                                                                  │
│ ---                                                                                              │
│ #### 1. Цели и задачи                                                                            │
│ 1. Сформировать комплексный профиль текущего состояния Тюмени по экономическим, социальным,      │
│ пространственным и инфраструктурным параметрам.                                                  │
│ 2. Выявить сильные и слабые стороны, зоны риска и возможности для стратегического развития.      │
│ 3. Подготовить основу для дальнейшего формирования концепции территориального планирования и     │
│ инвестиционных приоритетов.                                                                      │
│                                                                                                  │
│ ---                                                                                              │
│ #### 2. Структура исследования                                                                   │
│ | № | Раздел | Ключевые вопросы | Основные источники данных | Планируемые GIS‑слои |             │
│ |---|--------|------------------|--------------------------|----------------------|              │
│ | 1 | Социально‑экономический анализ | • Демографическая динамика (рост/сокращение, возрастная   │
│ структура, миграция).<br>• Производительность труда, структура ВГП, динамика отраслей.<br>•      │
│ Рынок труда (уровень безработицы, квалификационный профиль).<br>• Жилищный рынок (цены,          │
│ доступность, типы жилья).<br>• Коммерческая недвижимость (офисные, торговые площади).<br>•       │
│ Бюджетная обеспеченность (доходы, расходы, уровень муниципальных инвестиций). | • Росстат,       │
│ Тюменская областная статистика.<br>• Публичные отчёты Тюменского муниципального бюджета.<br>•    │
│ Платформы «Домофонд», «ЦИАН», «Агентство недвижимости».<br>• Опросы населения (при возможности). │
│ | • Демографический слой (население по районам).<br>• Слой ВГП/отраслей (по ТЭО).<br>• Слой      │
│ рынка труда (по уровням безработицы).<br>• Слой жилой/коммерческой недвижимости (ценовые         │
│ индексы). |                                                                                      │
│ | 2 | Пространственный анализ | • Функциональная организация территории (жилая, промышленная,    │
│ рекреационная, сельскохозяйственная).<br>• Структура землевладения (государственная,             │
│ муниципальная, частная, аренда).<br>• Природно‑экологический каркас (зоны охраны, водные         │
│ объекты, зоны риска).<br>• Архитектурно‑градостроительная среда (плотность застройки,            │
│ исторический центр, новые кварталы). | • Кадастровый реестр (ФГИС «Кадастр»).<br>• Геопортал     │
│ Тюменской области (слои ЗУ, охраняемых территорий).<br>• Публичные планы территориального        │
│ развития (ГТД, ПГТ).<br>• Спутниковые снимки (Sentinel‑2, Landsat). | • Слой ЗУ (по типу         │
│ владения).<br>• Слой функционального назначения (по ПГТ).<br>• Слой природных объектов (водоёмы, │
│ леса, зоны ООПТ).<br>• Слой плотности застройки (по кадастровым данным). |                       │
│ | 3 | Транспортный анализ | • Городской пассажирский транспорт (маршруты, частота,               │
│ покрытие).<br>• Внешний пассажирский и грузовой транспорт (железные дороги, автодороги,          │
│ аэропорт).<br>• Парковки (количество, расположение, доступность).<br>• Пешеходные зоны и         │
│ инфраструктура (тротуары, велодорожки). | • Данные Транспорт

╭────────────────────── 📌#c3108e by ce465c (Транспортный инженер‑аналитик) ───────────────────────╮
│ ### Запись № 2 (ID: ce465c‑01)                                                                   │
│ **Тема:** Транспортный вклад в диагностику экономического и пространственного состояния г.       │
│ Тюмень                                                                                           │
│ **Автор:** Транспортный инженер‑аналитик (ID ce465c)                                             │
│                                                                                                  │
│ ---                                                                                              │
│ #### 1. Цели и задачи                                                                            │
│ 1. Оценить текущий уровень доступности и эффективности городского и внешнего пассажирского и     │
│ грузового транспорта.                                                                            │
│ 2. Выявить узкие места в транспортной сети, их влияние на экономику, рынок труда и жилой фонд.   │
│ 3. Сформировать набор индикаторов транспортной доступности (ITA) и грузовой логистической        │
│ эффективности (TLE) для дальнейшего SWOT‑анализа.                                                │
│ 4. Подготовить рекомендации по оптимизации транспортных коридоров, развитию парковочной          │
│ инфраструктуры и расширению пешеходных/велосипедных зон.                                         │
│                                                                                                  │
│ ---                                                                                              │
│ #### 2. Необходимые данные (с указанием потенциальных источников)                                │
│ | № | Показатель | Источник | Примечание |                                                       │
│ |---|------------|----------|------------|                                                       │
│ | 1 | Маршруты, расписание, частота, вместимость автотранспорта (автобусы, троллейбусы,          │
│ маршрутки) | Департамент транспорта Тюмени, открытые данные ОТК (OpenTransport) | Гео‑привязка к │
│ остановочным пунктам, расчёт покрываемости (м²/чел) |                                            │
│ | 2 | Пассажиропоток по маршрутам (сутки/год) | Департамент транспорта, автоматизированные       │
│ системы учёта | Позволит построить индексы загрузки и выявить недоиспользованные линии |         │
│ | 3 | Данные о железнодорожных грузовых и пассажирских перевозках (объёмы, направления) | ОАО    │
│ «Тюменьжд», Росжелдор | Включить в анализ внешних грузовых коридоров |                           │
│ | 4 | Аэропорт Тюмень (кол-во рейсов, пассажиров, грузов) | ТюменьЭйрпорт, Росавиация | Оценка   │
│ воздушного доступа к региону |                                                                   │
│ | 5 | Автодорожная сеть (классы дорог, пропускная способность, уровень загруженности) |          │
│ Росавтодор, Яндекс/Google Traffic API | Тепловые карты загруженности, расчёт среднего времени в  │
│ пути |                                                                                           │
│ | 6 | Парковки (кол‑во мест, тип (платные/бесплатные), расположение, занятость) | Департамент    │
│ транспорта, муниципальные порталы, OpenStreetMap | Слой «парковки» → индексы доступности         │
│ парковочных мест |                                                                               │
│ | 7 | Пешеходные зоны, тротуары, велодорожки, инфраструктура «last‑mile» | Департамент           │
│ транспорта, GIS‑слои города, OpenStreetMap | Оценка качества пешеходного и велосипедного доступа │
│ к ТЦ, школам, больницам |                                                                        │
│ | 8 | Данные о ДТП (кол‑во, тяжесть, места) | ГИБДД, открытые статистические порталы | Выявление │
│ проблемных участков, корреляция с пропускной способностью | 

In [15]:
expert_agents[0].messages

[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nNone (no explicit user request was provided).\n\n## SUMMARY\n- Executed a web search for **“Tyumen population 2023”**. Returned snippets include:\n  - Wikipedia entry noting Tyumen’s historical population exceeding 30,000 by the late 19th\u202fcentury and rapid recent growth, with a 2021 Census figure of 847,488.\n  - Additional Wikipedia pages on Tyumen‑Tura Tatars and Tyumen Oblast demographics.\n  - A population‑statistics site (populationstat.com) with a 2026 projection graph.\n- Executed a web search for **“Tyumen unemployment rate 2023”**. Returned snippets include:\n  - General Wikipedia list of countries by unemployment rate (no Tyumen‑specific figure).\n  - Wikipedia entry on Tyumen mentioning its economic context but not a specific unemployment rate.\n  - Wikipedia page on Russian federal subjects by GDP per capita (relevant economic data, not unemployment).\n  - Wikipedia page on Tyum

In [8]:
role_agents[2].messages

[SystemMessage(content='Записей на доске: 0', additional_kwargs={}, response_metadata={}, id='c2a93d3f-1fb9-4162-8fe7-585a14336360'),
 AIMessage(content='{ "content": "### Запись №\u202f1 (ID: a39afb‑01)\\n**Тема:** План диагностики экономического и пространственного состояния г.\u202fТюмень\\n**Автор:** Региональный планировщик‑географ (ID\u202fa39afb)\\n\\n---\\n#### 1. Цели и задачи\\n1. Сформировать комплексный профиль текущего состояния Тюмени по экономическим, социальным, пространственным и инфраструктурным параметрам.\\n2. Выявить сильные и слабые стороны, зоны риска и возможности для стратегического развития.\\n3. Подготовить основу для дальнейшего формирования концепции территориального планирования и инвестиционных приоритетов.\\n\\n---\\n#### 2. Структура исследования\\n| № | Раздел | Ключевые вопросы | Основные источники данных | Планируемые GIS‑слои |\\n|---|--------|------------------|--------------------------|----------------------|\\n| 1 | Социально‑экономический анали

In [10]:
role_agents[0].messages

[SystemMessage(content='Записей на доске: 0', additional_kwargs={}, response_metadata={}, id='2218ddc8-7d98-488a-8d40-136d207f0a1b'),
 AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 463, 'total_tokens': 495, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-317', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d402e-0f44-7ef0-8dd8-1324139f9391-0', tool_calls=[{'name': 'get_notes', 'args': {}, 'id': 'call_7yx3qx6d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 463, 'output_tokens': 32, 'total_tokens': 495, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content=[], name='get_notes', id='9c5352f8-76ba-4134-bd2b-1f51a8c01aff', tool_call_id='call_7yx3qx6d'),
 AIMessage(content='', additional_

In [11]:
board.notes

[Note(content='### Запись №\u202f1 (ID: a39afb‑01)\n**Тема:** План диагностики экономического и пространственного состояния г.\u202fТюмень\n**Автор:** Региональный планировщик‑географ (ID\u202fa39afb)\n\n---\n#### 1. Цели и задачи\n1. Сформировать комплексный профиль текущего состояния Тюмени по экономическим, социальным, пространственным и инфраструктурным параметрам.\n2. Выявить сильные и слабые стороны, зоны риска и возможности для стратегического развития.\n3. Подготовить основу для дальнейшего формирования концепции территориального планирования и инвестиционных приоритетов.\n\n---\n#### 2. Структура исследования\n| № | Раздел | Ключевые вопросы | Основные источники данных | Планируемые GIS‑слои |\n|---|--------|------------------|--------------------------|----------------------|\n| 1 | Социально‑экономический анализ | • Демографическая динамика (рост/сокращение, возрастная структура, миграция).<br>• Производительность труда, структура ВГП, динамика отраслей.<br>• Рынок труда (ур

In [12]:
controller_agent.messages

[SystemMessage(content='Записей на доске: 0', additional_kwargs={}, response_metadata={}, id='97d55bce-6cf4-4a50-ba4c-01e4f4eb8b0d'),
 AIMessage(content='{\n  "agents_ids": [\n    "51f6e8", "2bf0fb", "a39afb", "ce465c"\n  ]\n}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 900, 'total_tokens': 937, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-902', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d402d-fd5a-7840-8cd0-597a1f225c7b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 900, 'output_tokens': 37, 'total_tokens': 937, 'input_token_details': {}, 'output_token_details': {}})]

### Summarize

In [13]:
summarizer_actor_agent = Agent(
    tools=board.get_ro_tools(),  # оставляем инструменты
    system_prompt="""
    Ваша задача: синтезировать финальный ответ на поставленную задачу, основываясь ТОЛЬКО на материалах доски.
    
    Поставленная задача:
    `{question}`
    
    Алгоритм:
    1. Вызовите get_notes(), чтобы увидеть все записи
    2. Вызовите get_note() для КАЖДОЙ из них
    3. Синтезируйте финальный ответ
    4. Верните ТОЛЬКО финальный ответ, без лишних комментариев
    """.format(question=board.question)
)

summarizer_critic_agent = Agent(
    tools=board.get_ro_tools(),  # оставляем инструменты
    system_prompt="""
    Ваша задача: на основе черновика финального ответа на поставленную задачу и материалов на доске выдайте список замечаний.
    
    Поставленная задача:
    `{question}`
    """.format(question=board.question)
)

In [14]:
messages = []

n_gen = 3

for i in tqdm(range(n_gen)):
    actor_res = summarizer_actor_agent.invoke([HumanMessage(m) for m in messages[-1:]])
    messages.append(actor_res)

    if (i+1) < n_gen:
        critic_res = summarizer_critic_agent.invoke([HumanMessage(m) for m in messages[-1:]])
        messages.append(critic_res)

  0%|          | 0/3 [01:50<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
messages

['**Целеполагание г. Гатчина (Ленинградская область)\u202fна 2026‑2035\u202fгоды (без количественных показателей)**  \n\n---\n\n### 1. Устойчивое развитие и экологическая безопасность  \n- Формировать городскую среду, ориентированную на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, переход автопарка муниципальных и частных организаций на экологически чистый транспорт.  \n- Увеличивать долю зелёных насаждений и открытых пространств: создание новых парков и рекреационных зон вдоль реки Ижоры, развитие линейных зелёных коридоров, интеграция вертикального озеленения в архитектуру.  \n- Внедрять принципы «зелёной» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.  \n\n### 2. Транспортная и логистическая интеграция  \n- Создать мультимодальный транспортный узел, соединяющий железнодорожный вокзал, автовокзал и будущие линии скоростного обще

In [ ]:
print(messages[-1])

**Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы  
(без количественных показателей)**  

---

### 1. Устойчивое развитие и экологическая безопасность  
- **Зелёная инфраструктура** – расширение сети парков, рекреационных зон вдоль реки Ижоры; создание линейных зелёных коридоров и вертикального озеленения в новых кварталах.  
- **Возобновляемая энергетика и энергетическая независимость** – стимулирование установки солнечных панелей, ветровых турбин и биогазовых установок; развитие муниципальных микросетей; поддержка «зелёных тарифов» и субсидий для частных владельцев ВИЭ.  
- **Энерго‑ и ресурсосбережение** – внедрение энергоэффективных технологий в зданиях, умного освещения и автоматизированного контроля потребления.  
- **Водоснабжение и качество воды** – интеграция систем сбора дождевой и серой воды в муниципальную сеть, повторное использование, мониторинг качества и программы снижения потерь.  

### 2. Транспорт и логистика  
- **Мультимодальный транспортный узел

In [ ]:
board.notes

[Note(id='905e2f', author_id='a9f900', author_role='Градостроительный планировщик', content='Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных показателей)\n\n1. **Устойчивое развитие и экологическая безопасность**\n   - Формирование городской среды, ориентированной на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.\n   - Увеличение доли зеленых насаждений и открытых пространств: создание новых парков, рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция вертикального озеленения в архитектуре.\n   - Внедрение принципов «зеленой» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.\n\n2. **Транспортная и логистическая интеграция**\n   - Развитие мультимодального транспортного 

In [ ]:
from src import llm

res = llm.invoke([
    SystemMessage(f"""
    Ваша задача: синтезировать финальный ответ на поставленную задачу, основываясь ТОЛЬКО на материалах доски.
    ---
    Поставленная задача:
    `{board.question}`
    ---
    Записи на доске:
    {str(board.notes)}
    """)
])

In [ ]:
board.notes

[Note(id='905e2f', author_id='a9f900', author_role='Градостроительный планировщик', content='Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных показателей)\n\n1. **Устойчивое развитие и экологическая безопасность**\n   - Формирование городской среды, ориентированной на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.\n   - Увеличение доли зеленых насаждений и открытых пространств: создание новых парков, рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция вертикального озеленения в архитектуре.\n   - Внедрение принципов «зеленой» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.\n\n2. **Транспортная и логистическая интеграция**\n   - Развитие мультимодального транспортного 

In [ ]:
print(res.content)

**Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы**  
*(без привязки к количественным показателям; ориентировано на качественные результаты)*  

---

### 1. Устойчивое развитие и экологическая безопасность  
- Формировать городскую среду, ориентированную на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.  
- Увеличивать долю зелёных насаждений и открытых пространств: создание новых парков и рекреационных зон вдоль реки Ижора, развитие линейных зелёных коридоров, интеграция вертикального озеленения в архитектуру.  
- Внедрять принципы «зелёной» инфраструктуры в новых районах: энерго‑эффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.  

### 2. Транспортная и логистическая интеграция  
- Развивать мультимодальный транспортный узел, соединяющий железнодорожный во